# 01 · Data Loading & TF-IDF Baseline

Loads Financial PhraseBank and Twitter Financial News Sentiment, produces
stratified 70/15/15 splits (seed 42), saves split metadata, and runs a
TF-IDF + Logistic Regression baseline.

**Writes:** `results/splits_metadata.json`, `results/main_comparison.json`

In [ ]:
!pip install -q 'datasets>=2.14,<4.0' scikit-learn numpy scipy

## 1 · Configuration

Set `USE_DRIVE = True` and update `DRIVE_PATH` when running on Colab
with Google Drive mounted.

In [ ]:
# ── Toggle this when running on Colab ──────────────────────────
USE_DRIVE  = False
DRIVE_PATH = '/content/drive/MyDrive/cs443-final-project'
# ───────────────────────────────────────────────────────────────

import json, random, time
import numpy as np
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path(DRIVE_PATH)
else:
    BASE = Path('.')

RESULTS_DIR = BASE / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print('Base path:', BASE.resolve())

## 2 · Label Scheme & Metrics Helpers

Unified label scheme used across every notebook: **0 = negative, 1 = neutral, 2 = positive**.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

ID2LABEL   = {0: 'negative', 1: 'neutral', 2: 'positive'}
LABEL2ID   = {v: k for k, v in ID2LABEL.items()}
LABEL_NAMES = ['negative', 'neutral', 'positive']


def compute_metrics(y_true, y_pred):
    y_true, y_pred = list(y_true), list(y_pred)
    acc      = float(accuracy_score(y_true, y_pred))
    macro_f1 = float(f1_score(y_true, y_pred, average='macro',
                               labels=[0, 1, 2], zero_division=0))
    per_f1   = f1_score(y_true, y_pred, average=None,
                        labels=[0, 1, 2], zero_division=0)
    cm       = confusion_matrix(y_true, y_pred, labels=[0, 1, 2]).tolist()
    return {
        'accuracy':         acc,
        'macro_f1':         macro_f1,
        'per_class_f1':     {ID2LABEL[i]: float(per_f1[i]) for i in range(3)},
        'confusion_matrix': cm,
    }


def print_metrics(m, title=''):
    if title:
        print('\n' + '─' * 54)
        print('  ' + title)
        print('─' * 54)
    acc = m['accuracy']
    f1  = m['macro_f1']
    print(f'  Accuracy : {acc:.4f}   Macro F1: {f1:.4f}')
    for name, val in m['per_class_f1'].items():
        print(f'  {name:<10}: F1={val:.4f}')

## 3 · Load Financial PhraseBank

Uses the Parquet mirror `gtfintechlab/financial_phrasebank_sentences_allagree`
which avoids the deprecated loading-script requirement of the original dataset.
All 2,264 examples (100% annotator agreement) are split 70 / 15 / 15
using stratified sampling so every split reflects the same class distribution.

In [ ]:
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split

print('Loading Financial PhraseBank (Parquet mirror)...')
raw = load_dataset('gtfintechlab/financial_phrasebank_sentences_allagree')
ds  = raw['train']

# Normalise column name to 'text' for consistency with Twitter dataset
if 'sentence' in ds.column_names:
    ds = ds.rename_column('sentence', 'text')

print(f'  Columns  : {ds.column_names}')
print(f'  Total    : {len(ds)} examples')
print(f'  Example  : {ds[0]}')

texts  = list(ds['text'])
labels = list(ds['label'])

# ── Stratified 70 / 15 / 15 split ──
train_texts, tmp_texts, train_labels, tmp_labels = train_test_split(
    texts, labels, test_size=0.30, random_state=SEED, stratify=labels
)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    tmp_texts, tmp_labels, test_size=0.50, random_state=SEED, stratify=tmp_labels
)

train_ds = Dataset.from_dict({'text': train_texts, 'label': train_labels})
val_ds   = Dataset.from_dict({'text': val_texts,   'label': val_labels})
test_ds  = Dataset.from_dict({'text': test_texts,  'label': test_labels})

for split, d in [('train', train_ds), ('val', val_ds), ('test', test_ds)]:
    counts = {ID2LABEL[i]: sum(1 for l in d['label'] if l == i) for i in range(3)}
    print(f'  {split:<6}: {len(d):4d} examples | {counts}')

## 4 · Load Twitter Financial News Sentiment (OOD)

Used only for out-of-distribution evaluation — never trained on.
The dataset's native label scheme (0=Bearish, 1=Bullish, 2=Neutral) is
remapped to our unified scheme. Downsampled to 1,000 stratified examples.

In [ ]:
# Twitter native: 0=Bearish, 1=Bullish, 2=Neutral
# Our scheme:     0=negative, 1=neutral,  2=positive
TWITTER_REMAP = {0: 0, 1: 2, 2: 1}

print('Loading Twitter Financial News Sentiment...')
raw_tw  = load_dataset('zeroshot/twitter-financial-news-sentiment')
tw_full = raw_tw['validation']
tw_full = tw_full.map(lambda ex: {'label': TWITTER_REMAP[ex['label']]})

# Stratified 1 000-example subsample
tw_labels = np.array(tw_full['label'])
rng = np.random.default_rng(SEED)
indices = []
for cls in range(3):
    cls_idx = np.where(tw_labels == cls)[0]
    n = round(1000 * len(cls_idx) / len(tw_labels))
    indices.extend(rng.choice(cls_idx, size=n, replace=False).tolist())
rng.shuffle(indices)
tw_ds = tw_full.select(indices[:1000])

tw_counts = {ID2LABEL[i]: sum(1 for l in tw_ds['label'] if l == i) for i in range(3)}
print(f'  Twitter OOD: {len(tw_ds)} examples | {tw_counts}')
print(f'  Example    : {tw_ds[0]}')

## 5 · Save Split Metadata

In [ ]:
def class_dist(labels_list):
    return {ID2LABEL[i]: sum(1 for l in labels_list if l == i) for i in range(3)}

metadata = {
    'label_scheme': ID2LABEL,
    'phrasebank': {
        'train': {'size': len(train_ds), 'class_distribution': class_dist(train_labels)},
        'val':   {'size': len(val_ds),   'class_distribution': class_dist(val_labels)},
        'test':  {'size': len(test_ds),  'class_distribution': class_dist(test_labels)},
    },
    'twitter_ood': {
        'size': len(tw_ds),
        'class_distribution': class_dist(tw_ds['label']),
    },
}

meta_path = RESULTS_DIR / 'splits_metadata.json'
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print('Saved:', meta_path)

## 6 · TF-IDF + Logistic Regression Baseline

Classical bag-of-words baseline. No embeddings, no pretraining — sets a
floor for what count-based features can achieve on financial text.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

print('Fitting TF-IDF vectoriser...')
vec = TfidfVectorizer(max_features=20_000, ngram_range=(1, 2), sublinear_tf=True)
X_train = vec.fit_transform(train_texts)
X_test  = vec.transform(test_texts)

print('Training Logistic Regression...')
clf = LogisticRegression(max_iter=1000, random_state=SEED, solver='lbfgs')
clf.fit(X_train, train_labels)

t0    = time.perf_counter()
y_pred = clf.predict(X_test).tolist()
elapsed = time.perf_counter() - t0

tfidf_metrics = compute_metrics(test_labels, y_pred)
print_metrics(tfidf_metrics, 'TF-IDF + LogReg — PhraseBank Test')
n = len(test_labels)
print(f'  Inference: {elapsed:.4f}s for {n} examples ({elapsed/n*1000:.2f} ms/example)')

## 7 · Save Results

In [ ]:
results_path = RESULTS_DIR / 'main_comparison.json'

# Load existing results if present (later notebooks append to this file)
if results_path.exists():
    with open(results_path) as f:
        all_results = json.load(f)
else:
    all_results = {}

all_results['tfidf_logreg'] = {
    'model': 'tfidf_logreg',
    'phrasebank_test': {
        **tfidf_metrics,
        'num_examples':      len(test_labels),
        'inference_seconds': round(elapsed, 4),
        'cost_per_1k':       0.0,
    },
}

with open(results_path, 'w') as f:
    json.dump(all_results, f, indent=2)

print('Saved:', results_path)
acc = tfidf_metrics['accuracy']
f1  = tfidf_metrics['macro_f1']
print(f'  TF-IDF + LogReg  acc={acc:.4f}  macro_f1={f1:.4f}')